In [3]:
import pandas as pd
import os

base_dir = "/hpc/group/kamaleswaranlab/mimic_iv/builtdata/csv_exports"
output_dir = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files"
os.makedirs(output_dir, exist_ok=True)

labitems = pd.read_csv(os.path.join(base_dir, "hosp_d_labitems.csv"))

In [4]:
# Output file
out_path = os.path.join(output_dir, "LABS.csv")
if os.path.exists(out_path):
    os.remove(out_path)
    
header_written = False
chunksize = 1_000_000  # process 1M rows at a time
total_written = 0

for chunk in pd.read_csv(os.path.join(base_dir, "hosp_labevents.csv"), chunksize=chunksize):
    labs = chunk.merge(labitems, on="itemid", how="left")

    labs_final = pd.DataFrame({
        "csn": labs["hadm_id"],
        "pat_id": labs["subject_id"],
        "component": labs["label"],
        "component_id": labs["itemid"],
        "lab_result": labs["valuenum"].combine_first(labs["value"]), # lab result: use valuenum if available, otherwise use value
        "lab_result_unit": labs["valueuom"],
        "lab_result_time": labs["storetime"],
        "collection_time": labs["charttime"],
        "result_status": "Final",
        "proc_cat_name": labs["fluid"],
        "proc_desc": labs["category"],
    })
    
    labs_final = labs_final.dropna(subset=["csn", "pat_id", "component_id"])

    # Append to CSV
    labs_final.to_csv(out_path, mode="a", index=False, header=not header_written)
    header_written = True

    total_written += len(labs_final)
    print(f"✅ Processed chunk, total rows written so far: {total_written}")

print(f"All done! Total rows written: {total_written}")
print(f"✅ New file has been saved to {out_path}")

✅ Processed chunk, total rows written so far: 549369
✅ Processed chunk, total rows written so far: 1072963
✅ Processed chunk, total rows written so far: 1597414
✅ Processed chunk, total rows written so far: 2137224
✅ Processed chunk, total rows written so far: 2660966
✅ Processed chunk, total rows written so far: 3190525
✅ Processed chunk, total rows written so far: 3708956
✅ Processed chunk, total rows written so far: 4252241
✅ Processed chunk, total rows written so far: 4770982
✅ Processed chunk, total rows written so far: 5311055
✅ Processed chunk, total rows written so far: 5850231
✅ Processed chunk, total rows written so far: 6384119
✅ Processed chunk, total rows written so far: 6930860
✅ Processed chunk, total rows written so far: 7487060
✅ Processed chunk, total rows written so far: 8037946
✅ Processed chunk, total rows written so far: 8551468
✅ Processed chunk, total rows written so far: 9083824
✅ Processed chunk, total rows written so far: 9610410
✅ Processed chunk, total rows